In [0]:
# Databricks notebook
# Purpose: Load the latest CSV files from Bronze CSV folder into Bronze Delta Tables

from pyspark.sql import functions as F
from pyspark.sql.types import StringType

#  ============================================================
# 1. CONFIGURATION
#  ============================================================

base_path_extracted_root = "/Volumes/workspace/susep/lakehouse/lakehouse-susep/bronze/raw/csv/"
base_path_delta = "/Volumes/workspace/susep/lakehouse/lakehouse-susep/bronze/delta/"

#SELECTED FILES 

files = [
    "Ses_campos.csv",
    "Ses_cias.csv",
    "Ses_dependencias.csv",
    "Ses_diversos.csv",
    "Ses_grupos_economicos.csv",
    "ses_gruposramos.csv",
    "Ses_ramos.csv",
    "Ses_seguros.csv",
    "SES_UF2.csv"
]

print(f"CSV base path: {base_path_extracted_root}")
print(f"ROOT base path: {base_path_delta}")

In [0]:
# ============================================================
# 2. GET LATEST FOLDER
# ============================================================
 
def get_latest_folder(base_path):
    folders = [
        item.path.rstrip("/").split("/")[-1]
        for item in dbutils.fs.ls(base_path)
        if item.isDir()
    ]
 
    if not folders:
        raise Exception(f"No folders found in path: {base_path}")
 
    latest_folder = max(folders)
 
    return base_path.rstrip("/") + "/" + latest_folder + "/"
 
base_path_csv = get_latest_folder(base_path_extracted_root)
 
print(f"Latest folder identified: {base_path_csv}")
 
batch_id = base_path_csv.rstrip("/").split("/")[-1]

In [0]:
#  ============================================================
# 2. CHECK IF BATCH EXISTS FUNCTION
#  ============================================================

#def batch_already_loaded(delta_path, batch_id):

#    try:
 #       df_existing = spark.read.format("delta").load(delta_path)
 #       return df_existing.filter(F.col("ingestion_batch_id") == batch_id).limit(1).count() > 0

  #  except:
  #      return False    # table does not exist yet

In [0]:
#  ============================================================
# 3.	INGEST FUNCTION
#  ============================================================

def ingest_csv_to_bronze(file_name):
	file_path = base_path_csv + file_name
	table_name = file_name.replace(".csv", "").lower()
	full_table_name = f"susep.bronze_{table_name}"
	delta_path = base_path_delta + table_name
	
	print(f"Starting ingestion for: {file_name}")
	
	# Check if batch already exists
	#if batch_already_loaded(delta_path, batch_id):
	#	print(f"Batch {batch_id} already loaded for {table_name}. Skipping...")
	#	return
	
	# Read CSV
	df = (
		spark.read.format("csv")
		.option("header", True)
		.option("sep", ";")
		.option("encoding", "latin1")
		.option("inferSchema", True)
		.load(file_path)
	)

	# Add technical columns
	df = (
		df
		.withColumn("ingestion_timestamp", F.current_timestamp())
		.withColumn("source_file", F.lit(file_name))
		.withColumn("source_file_path", F.lit(file_path))
		.withColumn("source_folder", F.lit(base_path_csv))
		.withColumn("ingestion_batch_id", F.lit(batch_id))
		.withColumn("source_table_name", F.lit(table_name))
		.withColumn("reference_year_month", F.lit(None).cast(StringType()))
		.withColumn("row_hash", F.sha2(F.concat_ws("||", *df.columns), 256))
	)

	# Write Delta (append + partition)
	(
		df.write
		.format("delta")
		.mode("append")
		.option("mergeSchema", "true")
		.partitionBy("ingestion_batch_id")
		.saveAsTable(full_table_name)
	)

	
	print(f"Ingestion completed: {table_name}")

In [0]:
# ============================================================
# 4. EXECUTION LOOP
# ============================================================
for file in files:
	ingest_csv_to_bronze(file)